# Practical Medical Record Pseudonymization with the Wowool SDK

This notebook walks through building a clinical text pseudonymization pipeline using the Wowool SDK. 

### Objectives:
1. Run a baseline out-of-the-box anonymization pass.
2. Define custom `.wow` pattern rules for medical record numbers, policy IDs, and age generalization.
3. Configure Python f-string formatters for consistent graph identifiers (`Patient_1`) and dynamic masking.
4. While we strip direct identifiers, we want to retain useful demographic indicators like gender and age for population-level studies
5. Audit the sanitized corpus for residual risk using the `UnknownThing` entity.


We will implement this pipeline using a combination of Wowool’s out-of-the-box entities and custom domain rules.

# Setup and Package Installation

In [1]:
# Install the required Wowool SDK packages
!pip install -q nlp-wowool-sdk wowool-anonymizer wowool-english wowool-snippet


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


### Set up your API key

In [2]:

import os
import json
from pathlib import Path

# Set your Wowool SDK API Key
os.environ["WOWOOL_SDK_KEY"] = "rNO/oPyNIqifrUXsQ3niwzXt4s3/iDI4B1AXt7H/5PSiayLfxmwLRK6FyJOFDDDjDoYHltbRbXyMdqzLdz45I80zkfKoBD3JVi0p5wyL6ZLur8H2JdGyZcK+0nGIyULqsskTReassgHUycPU3uurEEj6p6kU20a0F7KsQ++P5mm88a53bijjdNUgh3ytkRbO+vMtl3TS5L+poFpfM88WPUofWdSfvzxYqo+KIz6jcA1j/mRjOa9kLhhooojCFqba0rBa6pL/1Tak57IrUYc0s/gBH8gJ7uINLXiRUzQOxFgC4UmX/tRtfip81Pa/6kk8tjR/k36z0XijeSa/7Rm/W+NgMLzTL9LWDV0HJ9MvSXZQmfRvjPXsou9OBrixli2BVYLtPQMKWbYVX1sJhuYynmG6zWzYA9EeonhNIVJPVC/0uQcy2izKQPMqqtfjQfp4FZsVaxALGm/gkCobqV17h0pjE+RNZcW+z51gaMMwPxB4IWE3vF5+ybsplxeQwMy+z2aKZAhfQttvYs7VuAtdT6H+/u+qieTxDPTUe/ERgfs="
print("Wowool SDK API Key set successfully.")

Wowool SDK API Key set successfully.


Create Sample Medical Records

In [4]:
# Create directories for data, rules, and output
data_dir = Path("data")
data_dir.mkdir(parents=True, exist_ok=True)

file_path = data_dir / "sample_record.txt"

sample_record = """# MEDICAL REPORT - CONFIDENTIAL

Patient ID: MR-2024-789456
Date: March 15, 2024
Hospital: St. Mary's General Hospital
Department: Internal Medicine

## PATIENT INFORMATION:
Name: Sarah Johnson
DOB: 08/12/1978 (Age: 45)
Gender: Female
Address: 1247 Oak Street, Springfield, IL 62701
Phone: (217) 555-0198
Insurance: Blue Cross Blue Shield - Policy #: BC887456123

## ATTENDING PHYSICIAN:
Dr. Michael Chen, MD
Internal Medicine
License #: IL-MD-456789

## CLINICAL FINDINGS:
- Potassium: 4.2 mmol/L
- BP: 120/80 mmHg
- Assessment: Mild hypercholesterolemia, chronic obstructive pulmonary disease.
"""

with open(file_path, "w") as f:
    f.write(sample_record)

print("Sample record written to", file_path)

print("Working Directory:", os.getcwd())
print("Full Target Path: ", os.path.abspath(file_path))

Sample record written to data/sample_record.txt
Working Directory: /Users/crubio/dev/wowool/wowool-tutorials/notebooks/anonymizer-healthrecord
Full Target Path:  /Users/crubio/dev/wowool/wowool-tutorials/notebooks/anonymizer-healthrecord/data/sample_record.txt


## Baseline Anonymization

For this sample we will run the anonymizer driver using the built-in English and anonymizer pipelines. The same functionality can also be achieved using the Python scripts. (see example at: https://portal.wowool.com/docs/apps/anonymizer).



So run the driver with the following parameters:

-p: Specifies the linguistic pipeline (english,anonymizer).

-a: Lists the target annotation entities to anonymize.

-f: Target directory containing source files (.txt, .pdf, .docx, or .html).

In [8]:
# Run baseline anonymizer targeting relevant entities
!anonymizer -p english,anonymizer -a "Person,Address,Date,PhoneNr" -f "data/*.txt" --output_folder "output/anonymized"

# Inspect baseline output
print("=== Baseline Output ===")
with open("output/anonymized/sample_record.txt", "r") as f:
    print(f.read())

processing:/Users/crubio/dev/wowool/wowool-tutorials/notebooks/anonymizer-healthrecord/data/patient1.txt ...
file:/Users/crubio/dev/wowool/wowool-tutorials/notebooks/anonymizer-healthrecord/data/patient1.txt -> output/anonymized/patient1.txt
processing:/Users/crubio/dev/wowool/wowool-tutorials/notebooks/anonymizer-healthrecord/data/patient2.txt ...
file:/Users/crubio/dev/wowool/wowool-tutorials/notebooks/anonymizer-healthrecord/data/patient2.txt -> output/anonymized/patient2.txt
processing:/Users/crubio/dev/wowool/wowool-tutorials/notebooks/anonymizer-healthrecord/data/patient3.txt ...
file:/Users/crubio/dev/wowool/wowool-tutorials/notebooks/anonymizer-healthrecord/data/patient3.txt -> output/anonymized/patient3.txt
processing:/Users/crubio/dev/wowool/wowool-tutorials/notebooks/anonymizer-healthrecord/data/sample_record.txt ...
file:/Users/crubio/dev/wowool/wowool-tutorials/notebooks/anonymizer-healthrecord/data/sample_record.txt -> output/anonymized/sample_record.txt
=== Baseline Outp

This baseline leaves several gaps:

* Custom identifiers like Patient ID, Policy #, and License # were missed because they require domain-specific patterns.

* The anonymizer used a synthetic name (Marge Simpson) by default for Person, Company and Organization. For our graph database, we need a consistent identifier (e.g., Patient_1).

* Exact age values: for our purposes, we want to classify the patients in age groups, so we anonymize data, so that the identity of the patient cannot be deduced, but we keep important data for our analysis.


We can solve these issues by creating custom rules and customizing our formatters.

## Creating custom rules

To capture domain-specific patterns, create a directory called rules/ and add a .wow rules file (e.g., rules/healthrecord_anonymizer.wow):

> **Note**: the following cell cannot be run, it is a text file

In [ ]:
//---------------------------------------
// PatientID
// Patient IDs start with MR-
//---------------------------------------
rule: { "Patient ID" ":" {"MR-(.)*"}=PatientID };

//---------------------------------------
// Patient
// Create a new annotation just for patients
// Context: Name: John Doe
//---------------------------------------
rule: {"Name" ":" {Person}=Patient };

//---------------------------------------
// PolicyNumber
// Context: Policy #: DGASAS0090888
//---------------------------------------
rule: {"Policy" "#" ":" {<>}=PolicyNumber };

//---------------------------------------
// LicenseNumber
// Context: License #: PA-MD-123456
//---------------------------------------
rule: {"License" "#" ":" { (<>)+ }=LicenseNumber };

//---------------------------------------
// Age Categorization Rules
// 1–12            child
// 13–17           adolescent
// 18–24           young adult
// 25–44           adult
// 45–64           middle-aged adult
// 65+             older adult
//---------------------------------------

// Child (1 to 12 years old)
rule: { 
    Age[ 
        ("[:range(1-9):]" | "1[:range(0-2):]") 
    ] 
} = AgeGroup@(type="child");

// Adolescent (13 to 17 years old)
rule: {
    Age[ "1([:range(3-7):])" ] 
} = AgeGroup@(type="adolescent");

// Young Adult (18 to 24 years old)
rule: { 
    Age[ 
        ( "1(8|9)" | "2[:range(0-4):]" ) 
    ] 
} = AgeGroup@(type="young adult");

// Adult (25 to 44 years old)
rule: { 
    Age[ 
        ( "2[:range(5-9):]" | "3[:digit:]" | "4[:range(0-4):]" ) 
    ] 
} = AgeGroup@(type="adult");

// Middle-Aged Adult (45 to 64 years old)
rule: { 
    Age[ 
        ( "4[:range(5-9):]" | "5[:digit:]" | "6[:range(0-4):]" ) 
    ] 
} = AgeGroup@(type="middle-aged adult");

// Older Adult (65+ years old)
rule: { 
    Age[ 
        (
             "6[:range(5-9):]" 
             | "7[:digit:]" 
             | "8[:digit:]" 
             | "9[:digit:]" 
             | "10[:digit:]"
        ) 
    ] 
} = AgeGroup@(type="older adult");



**Custom rules** let us define domain-specific annotations (PatientID, AgeGroup, etc.) not detected by baseline models. Where possible, make rules as specific as the corpus allows to minimize false matches on unrelated text.

Append the rules directory to the pipeline flag so Wowool evaluates the custom rules alongside the default annotations. Add the newly created annotations behind the -a option (AgeGroup,PatientID,Patient,PolicyNumber,LicenseNumber).

In [10]:
!anonymizer -p english,anonymizer,rules -a "Person,Address,Date,PhoneNr,AgeGroup,PatientID,Patient,PolicyNumber,LicenseNumber" -f "data/*.txt" --output_folder "../output/anonymized"


with open("../output/anonymized/sample_record.txt", "r") as f:
    print(f.read())

processing:/Users/crubio/dev/wowool/wowool-tutorials/notebooks/anonymizer-healthrecord/data/patient1.txt ...
file:/Users/crubio/dev/wowool/wowool-tutorials/notebooks/anonymizer-healthrecord/data/patient1.txt -> ../output/anonymized/patient1.txt
processing:/Users/crubio/dev/wowool/wowool-tutorials/notebooks/anonymizer-healthrecord/data/patient2.txt ...
file:/Users/crubio/dev/wowool/wowool-tutorials/notebooks/anonymizer-healthrecord/data/patient2.txt -> ../output/anonymized/patient2.txt
processing:/Users/crubio/dev/wowool/wowool-tutorials/notebooks/anonymizer-healthrecord/data/patient3.txt ...
file:/Users/crubio/dev/wowool/wowool-tutorials/notebooks/anonymizer-healthrecord/data/patient3.txt -> ../output/anonymized/patient3.txt
processing:/Users/crubio/dev/wowool/wowool-tutorials/notebooks/anonymizer-healthrecord/data/sample_record.txt ...
file:/Users/crubio/dev/wowool/wowool-tutorials/notebooks/anonymizer-healthrecord/data/sample_record.txt -> ../output/anonymized/sample_record.txt
# MED

A few issues remain to be handled in the formatting layer:

* Replace fake doctor names with a generic label like Person_1.

* Mask unnecessary identifiers (e.g., license and policy numbers) with *** instead of semantic labels.

* Map exact age to the AgeGroup attribute created in our rule.

## Customizing the Formatters

Formatters define how matched entities appear in the output. Under the hood, they are evaluated as Python f-strings that expose Wowool objects, attributes, and standard Python string operations. 

Rather than accepting the defaults, we can specify programmatic structural replacements:

* Labels: Entity labels with counters: Patient_#{nr}.

* Entity Attributes & Python Methods: Access metadata directly and transform it on the fly (e.g., {concept.type.replace(" ", "_").upper()} to output MIDDLE_AGED_ADULT).

Dynamic Masking: Leverage Python expressions like len(literal) to build variable-length masks (#{"*" * len(literal)}#), or assign static replacements (#********#).